<a href="https://colab.research.google.com/github/Pea-hen/crypto-price-prediction/blob/main/crypto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
import matplotlib.pyplot as plt
!pip install mplfinance
import mplfinance as mpf

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
print("📊 Crypto Prediction System Started...")

In [ ]:
symbol = "BTC-USD"
raw_data = yf.download(symbol, period="60d", interval="1h")

market_data = raw_data[['Open', 'High', 'Low', 'Close']].copy()
market_data.dropna(inplace=True)

In [ ]:
# Moving Average
market_data['Short_MA'] = market_data['Close'].rolling(20).mean()
market_data['Long_MA'] = market_data['Close'].rolling(50).mean()

In [ ]:
# RSI
price_change = market_data['Close'].diff()
gain = price_change.clip(lower=0).rolling(14).mean()
loss = (-price_change.clip(upper=0)).rolling(14).mean()
rs_value = gain / loss
market_data['RSI_Value'] = 100 - (100 / (1 + rs_value))


In [ ]:
dataset = market_data.copy()

dataset['Hour'] = dataset.index.hour
dataset['Day'] = dataset.index.day
dataset['Month'] = dataset.index.month
dataset['Weekday'] = dataset.index.weekday

for lag in range(1, 25):
    dataset[f'Prev_{lag}'] = dataset['Close'].shift(lag)

dataset.dropna(inplace=True)

features = dataset.drop(columns=['Close'])
target = dataset['Close']


In [ ]:
cut_point = int(len(dataset) * 0.8)

train_X = features.iloc[:cut_point]
test_X  = features.iloc[cut_point:]

train_y = target.iloc[:cut_point]
test_y  = target.iloc[cut_point:]


In [ ]:
rf_model = RandomForestRegressor(n_estimators=300, random_state=1)
rf_model.fit(train_X, train_y)

test_output = rf_model.predict(test_X)

error = np.sqrt(mean_squared_error(test_y, test_output))
print(f"📉 Model RMSE: {error:.2f}")


In [ ]:
plt.figure(figsize=(12,6))
plt.plot(test_y.values, label="Real Price")
plt.plot(test_output, label="Model Prediction")
plt.title("Prediction vs Actual")
plt.xlabel("Time Index")
plt.ylabel("Price")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(market_data['Close'], label="Closing Price")
plt.plot(market_data['Short_MA'], label="MA 20")
plt.plot(market_data['Long_MA'], label="MA 50")
plt.title("Market Trend Analysis")
plt.legend()
plt.grid()
plt.show()


In [ ]:
user_time = input("\nEnter future date & time (YYYY-MM-DD HH:MM): ")
future_dt = datetime.strptime(user_time, "%Y-%m-%d %H:%M")

latest = dataset.iloc[-1:].copy()

latest['Hour'] = future_dt.hour
latest['Day'] = future_dt.day
latest['Month'] = future_dt.month
latest['Weekday'] = future_dt.weekday()

for lag in range(1, 25):
    latest[f'Prev_{lag}'] = dataset['Close'].iloc[-lag]


latest = latest.drop(columns=['Close'])

future_value = rf_model.predict(latest)[0]

print(f"\n📊 Estimated Price at {user_time}: {future_value:.2f}")
print("✅ Prediction completed successfully!")